# Task 2 — Fine-tuned vs. No-fine-tune Baseline

Compares the **fine-tuned** Llama-3.1-8B QLoRA run against the **un-fine-tuned base model** on Task 2 (Structured Entity Extraction), evaluated on the *same* validation set. The Task 1 counterpart is [finetune_vs_baseline_comparison.ipynb](finetune_vs_baseline_comparison.ipynb).

Inputs (both produced by Kaggle runs and downloaded via `kaggle/run.ps1`):

| Run | Notebook that produced it | Artifacts read here |
|---|---|---|
| Fine-tuned | `llm_fine_tuning_LORA_task2.ipynb` | `kaggle_output/eval_metrics.json` |
| Baseline (no fine-tune) | `llama_3.1_task_2_no_fine_tune.ipynb` | `kaggle_output_task2_baseline/no_finetune_baseline_task2/eval_metrics.json` |

The baseline notebook is an exact replica of the fine-tuning one with the training removed, and **runs the identical evaluation code** — same greedy generation, same `parse_prediction` / `norm` / `token_f1` / `value_f1`. Both build the validation set with the same contract-level `train_test_split(random_state=42)`, so the comparison is apples-to-apples; a cell below verifies that from the saved JSONL.

### The three metrics (full logic: [TASK2_EVALUATION_METRICS.md](docs/task_2/TASK2_EVALUATION_METRICS.md))

Unlike Task 1's Yes/No, a Task 2 answer can fail in three different ways, so each example is scored on three metrics **in a gating order** — an output that isn't valid JSON scores 0 on the other two by definition:

1. **JSON validity** — did the model obey the output contract at all (strict single-key JSON)?
2. **Exact match** — of well-formed outputs, is the value exactly the normalized gold value?
3. **Token F1** — partial credit for word overlap when the value is close but not exact.

This gives the per-example invariant `exact_match <= token_f1 <= json_valid`, which holds for the aggregates too and is asserted below.

Structure:
1. **Setup & loaders** — paths, palette, load both metric files, sanity checks.
2. **Same-validation-set proof** — record-by-record JSONL comparison.
3. **Headline comparison** — the three metrics, with deltas.
4. **Per-category breakdown** — all 9 entity categories, baseline vs. fine-tuned.
5. **Format vs. content diagnosis** — separating "couldn't emit JSON" from "emitted JSON with the wrong value". This is the analysis the three-metric design exists to support.
6. **Takeaways** — computed from the numbers, so they stay correct if you point at other runs.

## 1. Setup & loaders

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- Configuration: point these at any two runs with the standard Task 2 layout ----
FINETUNED_DIR = Path("kaggle_output")                                                   # fine-tuned run
BASELINE_DIR  = Path("kaggle_output_task2_baseline/no_finetune_baseline_task2")         # no-fine-tune baseline

# ---- Palette: one fixed color per model, everywhere in this notebook ----
# Same validated pair as the Task 1 comparison notebook, so the two read as one system.
# (CVD dE 73.6 deutan; the aqua is <3:1 on white, so every bar carries a visible value
#  label rather than relying on color alone.)
BASE_COLOR = "#1baf7a"   # baseline (aqua)
FT_COLOR   = "#2a78d6"   # fine-tuned (blue)
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"
SURFACE = "#fcfcfb"

plt.rcParams.update({
    "figure.figsize": (7, 4),
    "axes.grid": True, "grid.color": GRID,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#c3c2b7",
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.labelcolor": INK, "text.color": INK,
})

assert FINETUNED_DIR.exists(), f"Fine-tuned run dir not found: {FINETUNED_DIR.resolve()}"
assert BASELINE_DIR.exists(), f"Baseline run dir not found: {BASELINE_DIR.resolve()}"
print("Fine-tuned run:", FINETUNED_DIR.resolve())
print("Baseline run:  ", BASELINE_DIR.resolve())

In [ ]:
def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


base_metrics = load_json(BASELINE_DIR / "eval_metrics.json")
ft_metrics = load_json(FINETUNED_DIR / "eval_metrics.json")

# Fixed display order: baseline first, fine-tuned second — reused by every chart.
RUNS = {
    "Baseline (no fine-tune)": base_metrics,
    "Fine-tuned": ft_metrics,
}
RUN_COLORS = {"Baseline (no fine-tune)": BASE_COLOR, "Fine-tuned": FT_COLOR}

# The three metrics, in their gating order (see the header).
METRICS = ["json_valid", "exact_match", "f1"]
METRIC_LABELS = {"json_valid": "JSON validity", "exact_match": "Exact match", "f1": "Token F1"}

# Canonical category order (the order the training notebook defines them in).
CATEGORIES = [
    "Document Name", "Parties", "Agreement Date", "Effective Date",
    "Expiration Date", "Renewal Term", "Notice Period To Terminate Renewal",
    "Governing Law", "Warranty Duration",
]

# --- Sanity checks -----------------------------------------------------------
# Both runs must be Task 2, and must have been evaluated on the same validation set.
for name, m in RUNS.items():
    assert m.get("task") == "task2_entity_extraction", f"{name} is not a Task 2 run: {m.get('task')!r}"

assert base_metrics["n_validation_examples"] == ft_metrics["n_validation_examples"], (
    f"Different validation sizes: baseline={base_metrics['n_validation_examples']} "
    f"vs fine-tuned={ft_metrics['n_validation_examples']} — the runs are NOT comparable.")
assert base_metrics["per_category_counts"] == ft_metrics["per_category_counts"], (
    "Per-category example counts differ — the runs saw different validation data.")

# The gating invariant: exact_match <= token_f1 <= json_valid, per category and overall.
for name, m in RUNS.items():
    rows = list(m["per_category"].items()) + [("OVERALL", m["overall"])]
    for cat, r in rows:
        assert r["exact_match"] <= r["f1"] + 1e-9 <= r["json_valid"] + 1e-9, (
            f"{name} / {cat}: invariant exact_match <= f1 <= json_valid violated: {r}")
print("OK — gating invariant (exact_match <= token_f1 <= json_valid) holds in both runs.\n")

for name, m in RUNS.items():
    tuned = "no" if m.get("fine_tuned") is False else "yes"
    print(f"{name:>24}: {m['model_name']:<32} fine-tuned={tuned}  "
          f"({m['n_validation_examples']} validation examples)")

## 2. Same-validation-set proof

The size and per-category counts matching is necessary but not sufficient. Each run also saves the exact JSONL it evaluated on, so compare them **record by record** — if these differ, every number below is meaningless.

In [ ]:
def load_val_records(run_dir):
    path = run_dir / "cuad" / "validation" / "cuad_task2_validation.jsonl"
    if not path.exists():
        return None
    with open(path, encoding="utf-8") as f:
        return sorted(json.dumps(json.loads(line), sort_keys=True)
                      for line in f if line.strip())


base_val = load_val_records(BASELINE_DIR)
ft_val = load_val_records(FINETUNED_DIR)

if base_val is None or ft_val is None:
    print("WARNING: a validation JSONL is missing; falling back on the "
          "size/count checks above.")
elif base_val == ft_val:
    print(f"OK — both runs evaluated the identical {len(base_val)} validation examples.")
else:
    overlap = len(set(base_val) & set(ft_val))
    raise AssertionError(
        f"Validation sets DIFFER (only {overlap}/{len(base_val)} examples shared) — "
        "the metric comparison below would not be apples-to-apples.")

## 3. Headline comparison

The three metrics overall (micro-averaged across all 631 validation examples — categories with more examples weigh more). `Δ` is fine-tuned minus baseline, in percentage points.

In [ ]:
summary = pd.DataFrame(
    {name: {METRIC_LABELS[k]: m["overall"][k] for k in METRICS} for name, m in RUNS.items()}
).T
summary.loc["Δ (fine-tuned − baseline)"] = (
    summary.loc["Fine-tuned"] - summary.loc["Baseline (no fine-tune)"])

display(
    summary.style
    .format("{:.1%}", subset=(list(RUNS), slice(None)))
    .format("{:+.1f} pp", subset=(["Δ (fine-tuned − baseline)"], slice(None)))
    .set_caption("Task 2 overall metrics on the shared validation set")
)

In [ ]:
def bar_labels(ax, fmt="{:.0%}"):
    """Annotate each bar with its height, in text ink (never the series color)."""
    for p in ax.patches:
        h = p.get_height()
        ax.annotate(fmt.format(h), (p.get_x() + p.get_width() / 2, h),
                    ha="center", va="bottom", fontsize=9, color=INK)


x = np.arange(len(METRICS))
w = 0.38   # leaves a visible surface gap between the paired bars

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.set_axisbelow(True)   # grid behind the bars, never drawn across them
for i, (name, m) in enumerate(RUNS.items()):
    ax.bar(x + (i - 0.5) * (w + 0.02), [m["overall"][k] for k in METRICS],
           width=w, color=RUN_COLORS[name], label=name)

ax.set_xticks(x, [METRIC_LABELS[k] for k in METRICS])
ax.set_ylim(0, 1.15)
ax.set_ylabel("score")
ax.set_title("Task 2 overall — the metric gate, baseline vs. fine-tuned")
ax.legend(frameon=False, loc="upper right")
bar_labels(ax)
plt.tight_layout()
plt.show()

print("Read left-to-right: each metric is gated by the one before it. The baseline loses\n"
      "most of its score at the FIRST gate — it cannot reliably emit valid JSON at all.")

## 4. Per-category breakdown

Dates are short and highly templated; `Parties` is multi-valued free text. An aggregate would average an easy category against a hard one and hide exactly the structure worth reporting — so all 9 categories get their own row, with the validation example count in the label (a score over 11 examples is weaker evidence than one over 102).

In [ ]:
def per_category_frame(metric):
    return pd.DataFrame(
        {name: [m["per_category"][c][metric] for c in CATEGORIES] for name, m in RUNS.items()},
        index=CATEGORIES,
    )


counts = ft_metrics["per_category_counts"]
ylabels = [f"{c}  (n={counts[c]})" for c in CATEGORIES]

fig, axes = plt.subplots(1, 3, figsize=(14, 5.8), sharey=True)
y = np.arange(len(CATEGORIES))
h = 0.38

handles = None
for ax, metric in zip(axes, METRICS):
    ax.set_axisbelow(True)   # grid behind the bars
    data = per_category_frame(metric)
    for i, name in enumerate(RUNS):
        bars = ax.barh(y + (0.5 - i) * (h + 0.02), data[name], height=h,
                       color=RUN_COLORS[name], label=name)
        for b in bars:
            v = b.get_width()
            ax.annotate(f"{v:.0%}", (v, b.get_y() + b.get_height() / 2),
                        xytext=(3, 0), textcoords="offset points",
                        va="center", ha="left", fontsize=7.5, color=INK)
    ax.set_xlim(0, 1.28)
    ax.set_title(METRIC_LABELS[metric])
    ax.set_xlabel("score")
    ax.grid(axis="y", visible=False)
    handles = ax.get_legend_handles_labels()

axes[0].set_yticks(y, ylabels)
axes[0].invert_yaxis()

# Figure-level legend above the panels — inside any panel it would collide with the bars.
fig.legend(*handles, frameon=False, ncol=2, loc="upper center", bbox_to_anchor=(0.5, 0.95))
fig.suptitle("Per-category performance on the shared validation set", y=1.02)
plt.tight_layout(rect=(0, 0, 1, 0.93))
plt.show()

## 5. Format vs. content diagnosis

This is what the three-metric design is *for*. A low score has two very different causes, and the fix differs:

- **Format failure** — the model knows the answer but can't emit strict single-key JSON (markdown fences, prose, wrong key). Shows up as low `json_valid`.
- **Content failure** — the model emits perfectly valid JSON containing the wrong value. Shows up as high `json_valid` but low `exact_match`.

Separating them needs **exact match *conditional on* validity** (`exact_match / json_valid`): *of the outputs that actually parsed, what fraction had exactly the right value?* Both metrics are means over the same denominator, so the ratio is well-defined — except where `json_valid == 0`, in which case no output parsed and content simply **cannot be assessed** (reported separately rather than silently plotted as zero).

In [ ]:
def conditional_em(m):
    """Exact match GIVEN the output was valid JSON. NaN where nothing parsed."""
    out = {}
    for c in CATEGORIES:
        r = m["per_category"][c]
        out[c] = r["exact_match"] / r["json_valid"] if r["json_valid"] > 0 else np.nan
    return out


diag = pd.DataFrame({
    "baseline json_valid": {c: base_metrics["per_category"][c]["json_valid"] for c in CATEGORIES},
    "baseline EM | valid": conditional_em(base_metrics),
    "fine-tuned json_valid": {c: ft_metrics["per_category"][c]["json_valid"] for c in CATEGORIES},
    "fine-tuned EM | valid": conditional_em(ft_metrics),
}).loc[CATEGORIES]

display(
    diag.style
    .format("{:.0%}", na_rep="n/a (nothing parsed)")
    .set_caption("Format discipline (json_valid) vs. content accuracy given format (EM | valid)")
)

dead = [c for c in CATEGORIES if base_metrics["per_category"][c]["json_valid"] == 0]
if dead:
    print("Baseline produced ZERO valid JSON for: " + ", ".join(dead))
    print("-> For these, content accuracy is undefined: the base model never got past the format gate.")

In [ ]:
# Dumbbell chart: one row per category, baseline dot -> fine-tuned dot on each of the
# two diagnostic axes. The connector length IS the gain. (A scatter of format-vs-content
# would pile every fine-tuned point into the same corner — after tuning, json_valid is
# ~100% everywhere — and collide the labels; rows keep each category legible.)
panels = [
    ("json_valid",  "Format discipline\n(JSON validity)"),
    ("EM | valid",  "Content accuracy given format\n(exact match | valid)"),
]

fig, axes = plt.subplots(1, 2, figsize=(13, 5.6), sharey=True)
y = np.arange(len(CATEGORIES))

for ax, (key, title) in zip(axes, panels):
    ax.set_axisbelow(True)
    bvals = diag[f"baseline {key}"]
    fvals = diag[f"fine-tuned {key}"]
    for j, c in enumerate(CATEGORIES):
        b, f_ = bvals[c], fvals[c]
        if np.isnan(b):
            # Baseline emitted zero valid JSON -> content accuracy is undefined, not zero.
            ax.annotate("baseline: no valid JSON", (0.02, j), va="center", ha="left",
                        fontsize=7.5, color=MUTED, style="italic")
        else:
            ax.plot([b, f_], [j, j], color=GRID, lw=3, zorder=1, solid_capstyle="round")
            ax.scatter([b], [j], s=85, color=BASE_COLOR, zorder=3,
                       edgecolor=SURFACE, linewidth=1.5)
            ax.annotate(f"{b:.0%}", (b, j), xytext=(0, -13), textcoords="offset points",
                        ha="center", fontsize=7.5, color=INK)
        ax.scatter([f_], [j], s=85, color=FT_COLOR, zorder=3,
                   edgecolor=SURFACE, linewidth=1.5)
        ax.annotate(f"{f_:.0%}", (f_, j), xytext=(0, 9), textcoords="offset points",
                    ha="center", fontsize=7.5, color=INK)
    ax.set_xlim(-0.05, 1.12)
    ax.set_xlabel("score")
    ax.set_title(title)
    ax.grid(axis="y", visible=False)

axes[0].set_yticks(y, ylabels)
axes[0].invert_yaxis()

# Legend proxies — every dot is also positioned on a named row, so identity is never color-alone.
axes[0].scatter([], [], s=85, color=BASE_COLOR, label="Baseline (no fine-tune)")
axes[0].scatter([], [], s=85, color=FT_COLOR, label="Fine-tuned")
fig.legend(*axes[0].get_legend_handles_labels(), frameon=False, ncol=2,
           loc="upper center", bbox_to_anchor=(0.5, 0.95))
fig.suptitle("What fine-tuning fixed, per category — format (left) vs. content (right)", y=1.02)
plt.tight_layout(rect=(0, 0, 1, 0.92))
plt.show()

print("Left panel  = did it emit parseable JSON at all.\n"
      "Right panel = of the answers that DID parse, how many had exactly the right value.\n"
      "A long left-panel connector + short right-panel one = the model already knew the\n"
      "content and only lacked format discipline. The reverse = it had to learn extraction.")

## 6. Takeaways

Computed from the loaded metrics, so this stays correct when the run directories point elsewhere.

In [ ]:
b, f = base_metrics["overall"], ft_metrics["overall"]
n = ft_metrics["n_validation_examples"]

print(f"Validation set: {n} examples across {len(CATEGORIES)} entity categories\n")

for k in METRICS:
    print(f"  {METRIC_LABELS[k]:<14} {b[k]:>6.1%}  ->  {f[k]:>6.1%}   "
          f"({(f[k] - b[k]) * 100:+.1f} pp)")

print(f"\n1. FORMAT is what fine-tuning fixed first. JSON validity {b['json_valid']:.0%} -> "
      f"{f['json_valid']:.0%}: the base model fails the output contract on roughly "
      f"{1 - b['json_valid']:.0%} of examples, and every one of those scores 0 on the other\n"
      "   two metrics by definition — so the format gate alone caps its headline numbers.")

# Which categories were pure format failures for the baseline (knew content, couldn't emit JSON)?
knew = [c for c in CATEGORIES
        if base_metrics["per_category"][c]["json_valid"] > 0
        and not np.isnan(diag.loc[c, "baseline EM | valid"])
        and diag.loc[c, "baseline EM | valid"] >= 0.5]
if knew:
    print(f"\n2. The base model already KNEW the content for: {', '.join(knew)} — "
          "its exact match\n   among outputs that parsed is >=50%. There the fine-tune bought "
          "format discipline,\n   not knowledge.")

# Where it emitted fine JSON but the wrong value.
wrong = [c for c in CATEGORIES
         if base_metrics["per_category"][c]["json_valid"] >= 0.9
         and base_metrics["per_category"][c]["exact_match"] < 0.1]
if wrong:
    print(f"\n3. The opposite failure — well-formed JSON, wrong value — for: {', '.join(wrong)}. "
          "\n   The base model happily emits a parseable answer that is simply not the gold "
          "(normalized)\n   value; here the fine-tune had to teach the extraction/normalization itself.")

gap_ft = f["f1"] - f["exact_match"]
print(f"\n4. Fine-tuned Token F1 ({f['f1']:.0%}) still exceeds exact match ({f['exact_match']:.0%}) "
      f"by {gap_ft * 100:.0f} pp.\n   That gap is the residual 'right span, imperfect normalization' mass — "
      "the model finds the\n   correct entity but does not always render it in the exact gold form.")

worst = min(CATEGORIES, key=lambda c: ft_metrics["per_category"][c]["exact_match"])
wr = ft_metrics["per_category"][worst]
print(f"\n5. Hardest category after fine-tuning: '{worst}' "
      f"(exact match {wr['exact_match']:.0%}, but Token F1 {wr['f1']:.0%} "
      f"on {counts[worst]} examples)\n   — a large F1/EM gap means it finds the right values and misses "
      "on exact form/completeness,\n   not on comprehension.")